# TruthfulQA Split Generation & Belief-Conditioned Variants

This notebook builds the deterministic TruthfulQA dataset described in `truthfulqa_split_generation_agent_guide.md`.

**Pipeline**
1. Load `TruthfulQA.csv` and official `mc_task.json`
2. Parse reference answers, assign stable `question_id`s, join MC0/MC1/MC2 targets
3. Compute answer-richness diagnostics and balance splits **before** generating variants
4. Build belief candidates + matched triples (no Cartesian explosion)
5. Emit per-split JSONL tasks (`mc0` only for feature/optimization; `mc0/mc1/mc2` for validation/holdout)
6. Write manifests, reports, and run automated validation

**Split targets (790 questions)**

| Split | Questions |
| --- | ---: |
| `feature_selection` | 316 |
| `optimization` | 237 |
| `behavior_validation` | 118 |
| `holdout_test_behavior` | 119 |


## 0. Paths & configuration


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build_dataset import build_dataset
from src.build_splits import TARGET_COUNTS, split_quality_report
from src.parse_truthfulqa import load_truthfulqa_csv, add_richness_bins
from src.join_mc_targets import join_mc_targets
from src.build_beliefs import build_belief_candidates, build_belief_triples

DATA_RAW = ROOT / 'data_raw'
DATA_PROCESSED = ROOT / 'data_processed'
REPORTS = ROOT / 'reports'
CSV_PATH = DATA_RAW / 'TruthfulQA.csv'
MC_JSON_PATH = DATA_RAW / 'mc_task.json'

SPLIT_SEED = 42
BELIEF_TEMPLATE_VERSION = 'v1'

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('CSV exists:', CSV_PATH.exists(), '| MC JSON exists:', MC_JSON_PATH.exists())

## 1. Ensure raw inputs are present

Downloads official files from the [TruthfulQA repository](https://github.com/sylinrl/TruthfulQA) when missing.


In [ ]:
import urllib.request

RAW_CSV_URL = 'https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/TruthfulQA.csv'
RAW_MC_URL = 'https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/data/mc_task.json'

def ensure_download(url: str, dest: Path) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        print(f'OK  {dest.name} ({dest.stat().st_size} bytes)')
        return
    print(f'Downloading {url} -> {dest}')
    urllib.request.urlretrieve(url, dest)
    print(f'Saved {dest} ({dest.stat().st_size} bytes)')

ensure_download(RAW_CSV_URL, CSV_PATH)
ensure_download(RAW_MC_URL, MC_JSON_PATH)

## 2. Parse CSV, join official MC targets, compute richness

Methodological rules enforced here:
- split unit = original question (`question_id`)
- official `mc0/mc1/mc2` targets from JSON (no reconstructed distractors)
- answer-richness is measured **before** splitting


In [ ]:
questions = load_truthfulqa_csv(str(CSV_PATH))
questions = join_mc_targets(questions, str(MC_JSON_PATH))
questions = add_richness_bins(questions)

assert len(questions) == 790
assert questions['question_id'].is_unique
assert questions['mc0_eligible'].all() and questions['mc1_eligible'].all() and questions['mc2_eligible'].all()

richness_counts = questions['richness_bin'].value_counts().reindex(
    ['none', 'low', 'medium', 'rich']
).fillna(0).astype(int)

diagnostics = {
    'n_questions': len(questions),
    'richness_bin_counts': richness_counts.to_dict(),
    'n_min_alt_ge_1': int((questions['min_alt'] >= 1).sum()),
    'n_min_alt_ge_2': int((questions['min_alt'] >= 2).sum()),
    'n_incorrect_unique_ge_3': int((questions['n_incorrect_unique'] >= 3).sum()),
}
print(json.dumps(diagnostics, indent=2))
questions[['question_id', 'type', 'category', 'richness_bin', 'n_alt_correct', 'n_alt_incorrect', 'min_alt']].head()

### Answer-count distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
sns.histplot(questions['n_correct_unique'], bins=range(1, questions['n_correct_unique'].max() + 2),
             discrete=True, ax=axes[0], color='#4C78A8')
axes[0].set_title('Unique correct answers / question')
axes[0].set_xlabel('n_correct_unique')
sns.histplot(questions['n_incorrect_unique'], bins=range(1, questions['n_incorrect_unique'].max() + 2),
             discrete=True, ax=axes[1], color='#F58518')
axes[1].set_title('Unique incorrect answers / question')
axes[1].set_xlabel('n_incorrect_unique')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
order = ['none', 'low', 'medium', 'rich']
sns.barplot(x=order, y=[richness_counts[o] for o in order], ax=ax, color='#54A24B')
ax.set_title('Answer-richness bins (min_alt)')
ax.set_ylabel('questions')
for i, o in enumerate(order):
    ax.text(i, richness_counts[o] + 3, str(int(richness_counts[o])), ha='center')
plt.tight_layout()
plt.show()

## 3. Run full pipeline (split → beliefs → variants → validate)

Equivalent CLI:

```bash
python -m src.build_dataset \
  --csv data_raw/TruthfulQA.csv \
  --mc-json data_raw/mc_task.json \
  --output-dir data_processed \
  --split-seed 42 \
  --belief-template-version v1 \
  --semantic-filter-cache data_processed/semantic_filter.jsonl
```


In [ ]:
summary = build_dataset(
    csv_path=CSV_PATH,
    mc_json_path=MC_JSON_PATH,
    output_dir=DATA_PROCESSED,
    reports_dir=REPORTS,
    split_seed=SPLIT_SEED,
    belief_template_version=BELIEF_TEMPLATE_VERSION,
    semantic_filter_cache=DATA_PROCESSED / 'semantic_filter.jsonl',
    max_beliefs_per_polarity=None,
    allow_direct_beliefs=False,
    emit_leave_one_out=False,
    strict=True,
)

print('validation ok:', summary['validation']['ok'])
print('split counts:', summary['validation']['split_counts'])
print('eligible binary beliefs:', summary['validation']['n_eligible_binary'])
print('rejected beliefs:', summary['validation']['n_rejected'])
print('triples:', summary['validation']['n_triples'],
      '| questions with triples:', summary['validation']['n_questions_with_triples'])
print('variant files:')
for path, n in sorted(summary['variant_summary']['files'].items()):
    print(f'  {Path(path).relative_to(ROOT)}: {n} instances')

## 4. Split quality report

Answer-richness, Type, and Category are balanced across splits so evaluation sets are not preferentially answer-rich.


In [ ]:
manifest = pd.read_csv(DATA_PROCESSED / 'split_manifest.csv')
assert dict(manifest['split'].value_counts()) == TARGET_COUNTS

quality = summary['split_quality']
print('Max absolute proportion deviations vs full dataset:')
print(json.dumps(quality['max_abs_proportion_deviation'], indent=2))

rows = []
for split, info in quality['splits'].items():
    rows.append({
        'split': split,
        'n': info['question_count'],
        **{f'richness_{k}': info['richness_bin_counts'].get(k, 0) for k in ['none', 'low', 'medium', 'rich']},
        'mean_n_correct': round(info['n_correct_unique']['mean'], 2),
        'mean_n_incorrect': round(info['n_incorrect_unique']['mean'], 2),
    })
split_overview = pd.DataFrame(rows).set_index('split').loc[list(TARGET_COUNTS)]
split_overview

In [ ]:
dist = pd.read_csv(REPORTS / 'split_distribution.csv')
rich = dist[dist['variable'] == 'richness_bin'].copy()
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=rich, x='value', y='proportion', hue='split',
            order=['none', 'low', 'medium', 'rich'], ax=ax)
ax.set_title('Richness-bin proportions by split')
ax.set_ylabel('proportion')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

type_dist = dist[dist['variable'] == 'type'].copy()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=type_dist, x='value', y='proportion', hue='split', ax=ax)
ax.set_title('Type proportions by split')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Belief candidates & triples

- Reject unsuitable beliefs (`I have no comment`, meta refusals, duplicates, …)
- Binary eligibility requires semantic relation `equivalent` or `entails` vs the MC0 polarity target (deterministic judge; cacheable)
- Triples use `K_q = max(|C_q|, |I_q|)` with cycling — not the full Cartesian product
- Each question has total analysis weight 1 (`pair_weight = 1/K_q`)


In [ ]:
beliefs = pd.read_parquet(DATA_PROCESSED / 'belief_candidates.parquet')
triples = pd.read_parquet(DATA_PROCESSED / 'belief_triples.parquet')
rejected = pd.read_csv(REPORTS / 'rejected_beliefs.csv')

print('beliefs:', len(beliefs))
print('eligible_binary:', int(beliefs['eligible_binary'].sum()))
print('rejected:', len(rejected))
print('triples:', len(triples), '| questions:', triples['question_id'].nunique())
print('pair_weight sums (should be ~1 per question):',
      triples.groupby('question_id')['pair_weight'].sum().describe()[['min', 'mean', 'max']].to_dict())

print('\nRejection reasons:')
print(rejected['rejection_reason'].value_counts().head(10) if len(rejected) else '(none)')
print('\nSemantic relations (all candidates):')
print(beliefs['semantic_relation_to_mc0_target'].value_counts())
beliefs[['belief_id', 'question_id', 'polarity', 'is_best_answer', 'semantic_relation_to_mc0_target',
         'eligible_binary', 'rejection_reason']].head(8)

# Questions with no binary triples (typically Best Answer = "I have no comment"
# and no alternative correct belief entails the MC0 positive target)
q_ids = set(pd.read_parquet(DATA_PROCESSED / 'questions.parquet')['question_id'])
missing = sorted(q_ids - set(triples['question_id']))
print(f'\nQuestions without binary triples: {len(missing)} / {len(q_ids)}')
print('These remain in their assigned split but contribute no deference variants.')


## 6. Inspect generated task instances


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

fs_mc0 = load_jsonl(DATA_PROCESSED / 'feature_selection' / 'mc0.jsonl')
val_mc1 = load_jsonl(DATA_PROCESSED / 'behavior_validation' / 'mc1.jsonl')
hold_mc2 = load_jsonl(DATA_PROCESSED / 'holdout_test_behavior' / 'mc2.jsonl')

print('feature_selection mc0 instance:')
print(json.dumps({k: fs_mc0[0][k] for k in [
    'instance_id', 'question_id', 'split', 'format', 'belief_condition',
    'answer_order', 'prompt', 'option_a', 'option_b', 'correct_label',
    'question_weight', 'condition_weight', 'template_version'
]}, indent=2, ensure_ascii=False))

print('\nbehavior_validation mc1 formats present:',
      sorted({r['format'] for r in load_jsonl(DATA_PROCESSED / 'behavior_validation' / 'mc0.jsonl')}
             | {r['format'] for r in val_mc1}
             | {r['format'] for r in load_jsonl(DATA_PROCESSED / 'behavior_validation' / 'mc2.jsonl')}))
print('holdout mc2 targets (first instance):')
print(json.dumps(hold_mc2[0]['targets'][:4], indent=2, ensure_ascii=False), '...')

## 7. Acceptance checks (summary)

Outputs written under `data_processed/` and `reports/`:

```text
data_processed/
  questions.parquet
  split_manifest.csv
  belief_candidates.parquet
  belief_triples.parquet
  semantic_filter.jsonl
  feature_selection/mc0.jsonl
  optimization/mc0.jsonl
  behavior_validation/{mc0,mc1,mc2}.jsonl
  holdout_test_behavior/{mc0,mc1,mc2}.jsonl
reports/
  split_summary.json
  split_distribution.csv
  rejected_beliefs.csv
  validation_report.json
```


In [ ]:
validation = json.loads((REPORTS / 'validation_report.json').read_text())
assert validation['ok'], validation['errors']
assert validation['split_counts'] == TARGET_COUNTS

# Binary-only development splits
assert not (DATA_PROCESSED / 'feature_selection' / 'mc1.jsonl').exists()
assert not (DATA_PROCESSED / 'optimization' / 'mc2.jsonl').exists()

# Holdout sealed from development JSONL
holdout_ids = set(manifest.loc[manifest['split'] == 'holdout_test_behavior', 'question_id'])
for split in ('feature_selection', 'optimization', 'behavior_validation'):
    for rec in load_jsonl(DATA_PROCESSED / split / 'mc0.jsonl'):
        assert rec['question_id'] not in holdout_ids

print('All acceptance checks passed.')
print(json.dumps(validation, indent=2))